# figs/tables for CM MD-to-QM-cluster paper

**Please avoid working in the original version in the DeYonker-lab-scripts repo! Copy to your own QM dir and use your local version.**

<span style="color:red">**ALSO: TALK TO DOMI BEFORE DOING ANYTHING WITH THE SAMPLING STUFF**</span>

I'd recommend that this just be used for publication-level plots/tables, and keeping all the stuff about assessing quality of data/digging into results be kept in the plot_CM_results notebook (or a new separate one for compiling everyone's data and then messing around with it). I think it'll just be easier to keep final presentable results organised/separate from random stuff we're trying out.

This notebook pulls together everyone's data for the three active sites so it can all be plotted together.

|             | ligand | active site   | frames      | model type | parent directory for project                        |
|-------------|--------|---------------|-------------|------------|-----------------------------------------------------|
| **Dr D.**   | A:128  | A/C interface | all         | individual | /project/ndyonker/chem/cm-MD-processing/MD-QM/QM-A  |
| **Pedro**   | B:256  | B/A interface | all         | individual | /project/pssntngr/chem/chorismate_mutase/QM-B/      |
| **Haritha** | C:384  | C/B interface | all         | individual | /project/hksasi/chem/chorismate_mutase/QM-C/        |
| **Domi**    | all    | all           | every 100th | batch      | /project/dwappett/chorismate_mutase/QM-batch-models |


# Notebook environment setup


## import packages

In [1]:
import os, os.path, glob
import ast      # needed to properly process lists of imag modes from the csv
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import mpl_toolkits.axisartist as axisartist    # needed for centre-aligned y-axis labels in two-part version of FG heatmaps 
from matplotlib.lines import Line2D     # needed for making easy shared legends for all subplots
import seaborn as sns
import tol_colors as tc
import pickle

### a note about plotting packages

I generally recommend making scatter plots/histograms directly from the dataframes with the pandas plotting commands, and violin plots with the seaborn violinplot function. 

reasons why seaborn is MUCH better than matplotlib for violin plots:
1) they work even if your dataset has NaNs so you don't need to specifically filter them out (matplotlib won't plot a violin at all if there are any NaNs) 
2) the violin sizes can be scaled by the number of data points being plotted which is much more visually intuitive than them all being the same size. see [sampling subset size example](#subset_size_violins) (dunno why link jumps to after plot instead of subheading markdown cell where I put the tag but scroll up after you click)
3) the default settings are way nicer than matplotlib's so you don't need a bunch of extra commands to make them look good

the thing is though, seaborn is based around a different way of structuring data (long form vs wide form, see their explanation [here](https://seaborn.pydata.org/tutorial/data_structure.html)). instead of manually specifying x-axis positions of the violins/being able to plot violins one by one into the same set of axes like you can with matplotlib, you need to do some upfront work to reorganise the data into a long-form dataframe and potentially add in a new column that will be used to group the points into violins/determine their x-axis positions. but in my opinion, understanding what you need the data to look like is like 90% of the work here, and then writing the command to actually put it together is much easier. especially since you can reuse how I've already done it as a guide. 

## set font sizes/define paper-compatible fig widths

The following block sets up some default figure widths and font sizes to make it easier to make nice consistent figures for publication. We don't have to use them from day one but I figured it's nice to already have in here for later.

fig sizes for publications:
- single column: approx 8cm wide
- double column: approx 18cm wide


In [2]:
# set default font sizes
# use 16/18/20 for publication quality figures with widths of 8 or 18
SMALL_SIZE = 16 
MEDIUM_SIZE = 18 
BIGGER_SIZE = 20 

# these values are better when making smaller plots for viewing on screen 
#SMALL_SIZE = 12
#MEDIUM_SIZE = 12 
#BIGGER_SIZE = 16 

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

# single column figure width
scolw = 8
# double column figure width
dcolw = 18

# Read in data

## Read MD data

### Probe contacts for all frames

In [3]:
### read in batch analysis contact data
MDdata = {}
MDdata['A128_contacts'] = pd.read_pickle('/project/dwappett/old_MD_stuff/md/analysis_groupmeeting/MERGED-BATCH-A128-PROBE-WITH-WAT/batch_rawdf.pkl')
MDdata['B256_contacts'] = pd.read_pickle('/project/dwappett/old_MD_stuff/md/analysis_groupmeeting/MERGED-BATCH-B256-PROBE-WITH-WAT/batch_rawdf.pkl')
MDdata['C384_contacts'] = pd.read_pickle('/project/dwappett/old_MD_stuff/md/analysis_groupmeeting/MERGED-BATCH-C384-PROBE-WITH-WAT/batch_rawdf.pkl')

FG_dfs = {}

for i in ['A128_contacts','B256_contacts','C384_contacts']:
    MDdata[i] = MDdata[i]['p_tot']
    MDdata[i] = MDdata[i].unstack(level=1).T
    MDdata[i].index = ['f'+j.split('.')[1] for j in MDdata[i].index]
    prot = [j for j in MDdata[i].columns if 'WAT' not in j]
    FG_dfs[i] = MDdata[i].loc[:,prot]
    standardised_names = []
    for j in prot:
        ch, idx, res = j.split(':')
        if ch == 'A':
            idx = int(idx)
        elif ch == 'B':
            idx = int(idx) - 128
        elif ch == 'C':
            idx = int(idx) - 256
        if res.endswith('_SC'):
            res = 'SC_' + res.split('_')[0]     # put SC at start for consistency with other stuff/easier sorting
        standardised_names.append((idx, res))
    FG_dfs[i].columns = pd.MultiIndex.from_tuples(standardised_names)
    FG_dfs[i].index = pd.MultiIndex.from_tuples([(f,i.split('_')[0]) for f in FG_dfs[i].index])

FG_contact_df = pd.concat([FG_dfs['A128_contacts'],FG_dfs['B256_contacts'],FG_dfs['C384_contacts']],axis=0)
FG_contact_df.sort_index(axis=1,level=[0,1],ascending=[True,False],inplace=True)
FG_contact_df.sort_index(axis=0,level=0,inplace=True)
    

### cpptraj RMSDs

Note: All RMSDs used in this notebook (as of March 2026) have been calculated with cpptraj in /project/dwappett/old_MD_stuff/md/RMSDs_2026/.

Some of them are exactly the same as the ones used in Atsu's paper but it was much easier to just repeat them with all the new ones so all the results are together and clearly labelled.

In [4]:
### read in cpptraj RMSDs/clusters based on heavy RMSDs (S5)

MDdata['RMSDs'] = pd.read_csv('/project/dwappett/old_MD_stuff/md/RMSDs_2026/redo_CM_RMSDs_2026.dat',sep=r'\s+',index_col=0)
MDdata['RMSDs'].insert(0,'fnum',MDdata['RMSDs'].index.values)
MDdata['S5_clusters'] = pd.read_csv('/project/dwappett/old_MD_stuff/md/RMSDs_2026/redo_CM_S5clusters_2026.dat',sep=r'\s+',index_col=0)

## Read sampled sets of frames

### Atsu's original subsets

Note: these match the correction (DOI:10.1039/D6CP90040B), not the original paper: frames that did not meet the original RMSD criteria for S2/S3/S4 have been removed from those subsets.

Definitions of schemes (using the same names I've used for the RMSDs):
- S1: every 1000th frame
- S2: 'allMC_tofirst', filtered to subset within mean +/- 1 std, 30 randomly selected
    - 4 of original set did not actually meet criterion, removed here.
- S3: 'activeA_MC_tofirst', filtered to subset within mean +/- 1 std, 30 randomly selected
    - 5 of original set did not actually meet criterion, removed here.
- S4: 'activeA_SC_tofirst', filtered to subset within mean +/- 1 std, 30 randomly selected
    - 5 of original set did not actually meet criterion, removed here.
- S5: 'heavy_tofirst' + k-means clustering (3 clusters), 10 randomly selected from each cluster
    - actually ended up with 10 from C1 (0 in MDdata['S5_clusters']), 14 from C2 (2 in MDdata['S5_clusters']) and 6 from C3 (1 in MDdata['S5_clusters'])
- S6: 'activeA_MC_tofirst' + k-means clustering (4 clusters), 10 randomly selected from each cluster
- S7: 'activeA_SC_tofirst' + k-means clustering (4 clusters), 10 randomly selected from each cluster
- S8: probe contacts for all frames + k-means clustering (3 clusters), 10 randomly selected from each cluster
- S9: not in original paper, just grouping of the 14 frames removed from S2/S3/S4

In [5]:
### manually define Atsu's original sets ###
old_subsets = {}
old_subsets['S1'] = [1000,2000,3000,4000,5000,6000,7000,8000,9000,10000,11000,12000,13000,14000,15000,16000,17000,18000,19000,20000]
old_subsets['S2'] = [2373,2391,4688,5084,8089,8589,9278,9750,9855,10505,10657,10952,11115,11254,11818,12122,12836,12915,13103,14980,17651,18802,19126,19270,19719,19962]
old_subsets['S3'] = [1028,1368,1603,3436,4909,5874,6315,6849,6944,8299,10906,11047,11808,13719,13942,14120,14253,16450,16805,17235,17661,18009,18167,19441,19787]
old_subsets['S4'] = [336,389,929,956,1485,1617,2747,2862,3858,4448,4591,6011,6771,7128,7438,7457,8007,10580,11912,12749,13226,13445,13888,14064,17422]
old_subsets['S5'] = [174,703,1341,3483,3670,3713,3987,5133,5870,6271,7550,7607,7724,8310,8496,9449,9591,10084,11260,11417,12379,13402,13826,14007,14455,15726,16554,17301,18929,19615]
old_subsets['S6'] = [146,242,1169,1218,2475,2752,3059,3374,4114,4777,5556,5649,6086,6418,6570,7282,7311,7603,8157,8320,9104,9221,9464,9748,10287,10737,11363,11586,12180,12378,14168,14224,15805,16184,16652,17413,18312,18873,19247,19540]
old_subsets['S7'] = [161,183,198,220,227,291,324,363,370,394,729,827,1005,1221,1658,4202,5223,5690,5836,6678,6981,7364,8457,8717,9318,9366,9619,10854,10927,11171,11944,12969,13854,14177,15106,15838,16923,18212,19721,19921]
old_subsets['S8'] = [159,547,970,2150,2224,2473,3595,3808,4070,6464,7505,7967,8014,8677,9216,10960,11553,12405,13866,14855,15124,15169,15355,15461,15687,16603,16937,17536,18015,18531]
old_subsets['S9'] = [101,2322,15246,17817,758,4165,6412,12729,18630,232,806,8886,9662,18831]
old_subsets['all_250'] = old_subsets['S1']+old_subsets['S2']+old_subsets['S3']+old_subsets['S4']+old_subsets['S5']+old_subsets['S6']+old_subsets['S7']+old_subsets['S8']+old_subsets['S9']

# reformat into fxxxxx labels rather than integers
for key in old_subsets.keys():
    old_subsets[key] = [f'f{str(i).zfill(5)}' for i in old_subsets[key]]

### New sampled sets of frames

**Important: Please do NOT incorporate the actual sampling/generation of subsets into this notebook**

Otherwise you'll be regenerating the frame subsets every time, and then you'd need to be realllllly careful to make sure that you're a) always saving the current lists of sampled frames with any figures/tables made from that data, and b) updating _everything_ related to that sampling in the paper every time. E.g. if you've got a figure in the paper showing a sampled subset and then a table in the SI with the list of all models in that subset, then any time you tweaked the figure you'd need to also update the table so that the lists are consistent. Even if you have code in here to generate figures and latex tables at the same time, you still have to remember to move both to the manuscript every time which creates more opportunities for mistakes.

By generating the sampled subsets externally and just reading them in here, they will stay consistent and then you can deal with figures and tables separately with much less risk of things getting out of sync and having annoying mistakes in the results.

**Do the sampling with the external script here (again, talk to Domi before doing this): /project/dwappett/old_MD_stuff/md/RMSDs_2026/RMSD_sampling.py**

In [6]:
### read in sampled subsets ###
with open('/project/dwappett/old_MD_stuff/md/RMSDs_2026/sampled_sets.pkl','rb') as fp:
    sampled_sets = pickle.load(fp)

subsets that the RMSD_sampling.py script generates:
- 'interval_sampling': each has 5 replicas (from equally spaced starting points)
    - 'every1000th' / 'every500th' / 'every200th' / 'every100th' 
- RMSD-based sampling (from subset within mean +/- 1 std): each has 10 replicas for each of 'sample_10' / 'sample_20' / 'sample_30' / 'sample_40' / 'sample_50' / 'sample_100' / 'sample_150' / 'sample_200' / 'sample_250' / 'sample_300' 
    - 'allMC_tofirst' / 'allMC_toxtal'
    - 'activeA_MC_tofirst' / 'activeA_MC_toxtal'
    - 'activeA_SC_tofirst' / 'activeA_SC_toxtal'
    - 'activeB_MC_tofirst' / 'activeB_MC_toxtal'
    - 'activeB_SC_tofirst' / 'activeB_SC_toxtal'
    - 'activeC_MC_tofirst' / 'activeC_MC_toxtal'
    - 'activeC_SC_tofirst' / 'activeC_SC_toxtal'
    - 'activeMC_tofirst' / 'activeMC_toxtal'
    - 'activeSC_tofirst' / 'activeSC_toxtal'
    - 'heavy_tofirst' / 'heavy_toxtal'
- S5 cluster sampling: each has 10 replicas for each of 'sample_30' / 'sample_60' / 'sample_90' / 'sample_120' / 'sample_150' / 'sample_300' (10/20/30/40/50/100 from each cluster)
    - 'heavy_tofirst_cluster' / 'heavy_toxtal_cluster'

## Read QM data

In [7]:
QMdata = {}

# get data for active site A 
pathA = '/project/ndyonker/chem/cm-MD-processing/MD-QM/QM-A'
QMAfiles = glob.glob(f'{pathA}/*/*_results.csv')
if QMAfiles:
    QMAfiles.sort()
    for f in QMAfiles:
        if QMAfiles.index(f)==0:
            QMdata['A_ind'] = pd.read_csv(f,index_col='frame')
        else:
            QMdata['A_ind'] = pd.concat([QMdata['A_ind'],pd.read_csv(f,index_col='frame')])

# get data for active site B 
pathB = '/project/pssntngr/chem/chorismate_mutase/QM-B/'
QMBfiles = glob.glob(f'{pathB}/*/*_results.csv')
if QMBfiles:
    QMBfiles.sort()
    for f in QMBfiles:
        if QMBfiles.index(f)==0:
            QMdata['B_ind'] = pd.read_csv(f,index_col='frame')
        else:
            QMdata['B_ind'] = pd.concat([QMdata['B_ind'],pd.read_csv(f,index_col='frame')])

# get data for active site C
pathC = '/project/hksasi/chem/chorismate_mutase/QM-C/'
QMCfiles = glob.glob(f'{pathC}/*/*_results.csv')
if QMCfiles:
    QMCfiles.sort()
    for f in QMCfiles:
        if QMCfiles.index(f)==0:
            QMdata['C_ind'] = pd.read_csv(f,index_col='frame')
        else:
            QMdata['C_ind'] = pd.concat([QMdata['C_ind'],pd.read_csv(f,index_col='frame')])

# get data for batch models
pathbatch = '/project/dwappett/chorismate_mutase/QM-batch-models'
QMbatchfiles = glob.glob(f'{pathbatch}/*/*_results.csv')
if QMbatchfiles:
    QMbatchfiles.sort()
    for f in QMbatchfiles:
        if 'A128' in f:
            QMdata['A_batch']=pd.read_csv(f,index_col='frame')
        elif 'B256' in f:
            QMdata['B_batch']=pd.read_csv(f,index_col='frame')
        elif 'C384' in f:
            QMdata['C_batch']=pd.read_csv(f,index_col='frame')

# processing imag modes lists
for key in QMdata.keys():
    for struc in ['ts','r','p']:
        imagmodes = list(QMdata[key][f'{struc}_imagmodes'])
        ### this is only needed to fix the code while Pedro has un-updated results. vals might not actually be empty!!! remove later ###
        if key == 'B_ind': imagmodes = ['[]' if i != i else i for i in imagmodes]
        ############
        imagmodes = [ast.literal_eval(i) for i in imagmodes]
        QMdata[key][f'{struc}_imagmodes'] = imagmodes
        QMdata[key][f'{struc}_maximagmode'] = [i[0] if i else np.nan for i in imagmodes]
        nimag = list(QMdata[key][f'{struc}_Nimag'])
        QMdata[key][f'{struc}_Nimag'] = [int(i.replace('Nimag=','')) if isinstance(i,str) else i for i in nimag]
    QMdata[key]['tot_irc_Nimag'] = QMdata[key]['r_Nimag']+QMdata[key]['p_Nimag']


## Read model FG tables

### read original tables

In [8]:
model_FGs = {}

# get data for active site A 
pathA = '/project/ndyonker/chem/cm-MD-processing/MD-QM/QM-A'
QMAfiles = glob.glob(f'{pathA}/*/*_model_FGs.csv')
if QMAfiles:
    QMAfiles.sort()
    for f in QMAfiles:
        if QMAfiles.index(f)==0:
            model_FGs['A_ind'] = pd.read_csv(f,index_col='frame')
        else:
            model_FGs['A_ind'] = pd.concat([model_FGs['A_ind'],pd.read_csv(f,index_col='frame')])

# get data for active site B 
pathB = '/project/pssntngr/chem/chorismate_mutase/QM-B/'
QMBfiles = glob.glob(f'{pathB}/*/*_model_FGs.csv')
if QMBfiles:
    QMBfiles.sort()
    for f in QMBfiles:
        if QMBfiles.index(f)==0:
            model_FGs['B_ind'] = pd.read_csv(f,index_col='frame')
        else:
            model_FGs['B_ind'] = pd.concat([model_FGs['B_ind'],pd.read_csv(f,index_col='frame')])

# get data for active site C
pathC = '/project/hksasi/chem/chorismate_mutase/QM-C/'
QMCfiles = glob.glob(f'{pathC}/*/*_model_FGs.csv')
if QMCfiles:
    QMCfiles.sort()
    for f in QMCfiles:
        if QMCfiles.index(f)==0:
            model_FGs['C_ind'] = pd.read_csv(f,index_col='frame')
        else:
            model_FGs['C_ind'] = pd.concat([model_FGs['C_ind'],pd.read_csv(f,index_col='frame')])

# get data for batch models
pathbatch = '/project/dwappett/chorismate_mutase/QM-batch-models'
QMbatchfiles = glob.glob(f'{pathbatch}/*/*_model_FGs.csv')
if QMbatchfiles:
    QMbatchfiles.sort()
    for f in QMbatchfiles:
        if 'A128' in f:
            model_FGs['A_batch']=pd.read_csv(f,index_col='frame')
        elif 'B256' in f:
            model_FGs['B_batch']=pd.read_csv(f,index_col='frame')
        elif 'C384' in f:
            model_FGs['C_batch']=pd.read_csv(f,index_col='frame')

# add FG counts to main QMdata dataframes
for key in ['A_ind','B_ind','C_ind','A_batch','B_batch','C_batch']:
    QMdata[key]['nFG_all'] = model_FGs[key].sum(axis=1)
    QMdata[key]['nFG_prot'] = model_FGs[key][[c for c in model_FGs[key].columns if 'WAT' not in c]].sum(axis=1)
    QMdata[key]['nFG_SC'] = model_FGs[key][[c for c in model_FGs[key].columns if 'SC' in c]].sum(axis=1)
    QMdata[key]['nFG_MC'] = model_FGs[key][[c for c in model_FGs[key].columns if 'MC' in c]].sum(axis=1)
    QMdata[key]['nFG_wat'] = model_FGs[key][[c for c in model_FGs[key].columns if 'WAT' in c]].sum(axis=1)


### normalising residue names

In [ ]:
## dataframe with (frame,activesite) index and columns standardised to 1-127 res numbers (no waters) so can be directly compared

FG_dfs = {}
FG_equivalents = {}
#for model_set in ['A_ind','B_ind','C_ind']:
for model_set in ['A_ind','B_ind','C_ind','A_batch','B_batch','C_batch']:
    prot = [i for i in model_FGs[model_set].columns if 'WAT' not in i]
    FG_dfs[model_set] = model_FGs[model_set].loc[:,prot]
    FG_equivalents[model_set] = {}
    standardised_names = []
    for i in prot:
        ch, idx, res = i.split(':')
        idx = int(idx) % 128
        # if ch == 'A':
        #     idx = int(idx)
        # elif ch == 'B':
        #     idx = int(idx) - 128
        # elif ch == 'C':
        #     idx = int(idx) - 256
        standardised_names.append((idx, res))
        FG_equivalents[model_set][(idx, res)] = i
    FG_dfs[model_set].columns = pd.MultiIndex.from_tuples(standardised_names)
    FG_dfs[model_set].index = pd.MultiIndex.from_tuples([(f,model_set) for f in FG_dfs[model_set].index])

#FG_df = pd.concat([FG_dfs['A_ind'],FG_dfs['B_ind'],FG_dfs['C_ind']],axis=0)
FG_df = pd.concat([FG_dfs['A_ind'],FG_dfs['B_ind'],FG_dfs['C_ind'],FG_dfs['A_batch'],FG_dfs['B_batch'],FG_dfs['C_batch']],axis=0)

FG_df.sort_index(axis=1,level=[0,1],ascending=[True,False],inplace=True)
FG_df.sort_index(axis=0,level=0,inplace=True)

FG_df


## Define plotopts for axes lims/labels/etc

In [10]:
# define labels, max and min values so that we can easily set axes titles/legend values and axes limits consistently in plots
# keys of this dict are the column labels from the csv file/dataframes

plotopts = {'dGa': {'longlabel': r'$\Delta G ^\ddagger$ (kcal/mol)',
                'shortlabel': r'$\Delta G ^\ddagger$',
                'max': max([QMdata[key]['dGa'].max() for key in QMdata.keys()]), 
                'min': min([QMdata[key]['dGa'].min() for key in QMdata.keys()])
                },
        'dGr': {'longlabel': r'$\Delta G _{rxn}$ (kcal/mol)',
                'shortlabel': r'$\Delta G _{rxn}$',
                'max': max([QMdata[key]['dGr'].max() for key in QMdata.keys()]), 
                'min': min([QMdata[key]['dGr'].min() for key in QMdata.keys()])
                },
        'size': {'longlabel': 'model size (no. atoms)',
                'shortlabel': 'size',
                'max': max([QMdata[key]['size'].max() for key in QMdata.keys()]), 
                'min': min([QMdata[key]['size'].min() for key in QMdata.keys()])
                },
        'charge': {'longlabel': 'overall charge',
                'shortlabel': 'charge',
                'max': max([QMdata[key]['charge'].max() for key in QMdata.keys()]), 
                'min': min([QMdata[key]['charge'].min() for key in QMdata.keys()])
                },
        'fnum': {'longlabel': 'frame used to create model',
                'shortlabel': 'frame',
                'max': max([QMdata[key]['fnum'].max() for key in QMdata.keys()]),
                'min': min([QMdata[key]['fnum'].min() for key in QMdata.keys()])
                },
        'nFG_all': {'longlabel': 'no. FGs in model',
                'shortlabel': 'no. FGs',
                'max': max([QMdata[key]['nFG_all'].max() for key in QMdata.keys()]), 
                'min': min([QMdata[key]['nFG_all'].min() for key in QMdata.keys()])
                },
        'nFG_prot': {'longlabel': 'no. protein FGs in model',
                'shortlabel': 'no. protein FGs',
                'max': max([QMdata[key]['nFG_prot'].max() for key in QMdata.keys()]), 
                'min': min([QMdata[key]['nFG_prot'].min() for key in QMdata.keys()])
                },
        'nFG_SC': {'longlabel': 'no. SC fragments in model',
                'shortlabel': 'no. SCs',
                'max': max([QMdata[key]['nFG_SC'].max() for key in QMdata.keys()]), 
                'min': min([QMdata[key]['nFG_SC'].min() for key in QMdata.keys()])
                },
        'nFG_MC': {'longlabel': 'no. MC fragments in model',
                'shortlabel': 'no. MCs',
                'max': max([QMdata[key]['nFG_MC'].max() for key in QMdata.keys()]), 
                'min': min([QMdata[key]['nFG_MC'].min() for key in QMdata.keys()])
                },
        'nFG_wat': {'longlabel': 'no. waters in model',
                'shortlabel': 'no. waters',
                'max': max([QMdata[key]['nFG_wat'].max() for key in QMdata.keys()]), 
                'min': min([QMdata[key]['nFG_wat'].min() for key in QMdata.keys()])
                }
        }

ligands = ['A:128','B:256','C:384']
active_sites = ['A/C*','B/A*','C/B*']

# Results part 1: Composition of QM-cluster models

## prep: create table of FG info

In [ ]:
### table of FG info ###

# no. frames with contacts
df1 = FG_contact_df.groupby(level=1).count()*100/20000
df1.index = pd.MultiIndex.from_tuples([('% contacts',i) for i in df1.index])

# mean contact count
df2 = FG_contact_df.groupby(level=1).mean()
df2.index = pd.MultiIndex.from_tuples([('mean contacts',i) for i in df2.index])

# no. individual models where present
df3 = FG_df.groupby(level=1).sum().div(FG_df.groupby(level=1).size(),axis=0).loc[['A_ind','B_ind','C_ind'],:]*100
### uncomment below if you want FGs that are in actually 0 models of that active site to be NaNs so that they're distinct from ones that are in so few they're getting rounded to 0 ###
#sumdf = FG_df.groupby(level=1).sum()
#for i in ['A_ind','B_ind','C_ind']:
#    for j in df3.columns.values:
#        if sumdf.loc[i,j] == 0:
#            df3.loc[i,j] = np.nan
df3.index = pd.MultiIndex.from_tuples([('% ind mods','A128'),('% ind mods','B256'),('% ind mods','C384')])
# alternative version where it's number of models once we've got all 20000?
df3a = FG_df.groupby(level=1).sum().loc[['A_ind','B_ind','C_ind'],:]
df3a.index = pd.MultiIndex.from_tuples([('no. ind mods','A128'),('no. ind mods','B256'),('no. ind mods','C384')])

# presence in batch model
df4 = FG_df.loc[('f00100',['A_batch','B_batch','C_batch']),:].droplevel(0)
df4.index = pd.MultiIndex.from_tuples([('in batch mods','A128'),('in batch mods','B256'),('in batch mods','C384')])

FG_info = pd.concat([df1,df2,df3,df3a,df4]).T.round(2)
FG_info.sort_index(axis=0,level=[0,1],ascending=[True,False],inplace=True)

if ('fnum','') in list(FG_info.index.values):
    FG_info = FG_info.drop('fnum')
FG_info

## fig: FG heatmaps

Got both % contacts across all 20000 as well as % inclusion for all currently collected models right now because I think it's interesting for comparison.
For final paper we probably only want the % inclusion in models, that's directly after this.

In [ ]:
### heatmaps of models. might only need the % QM cluster models side for paper but interesting if looking at partial results to see how things compare
# also this is where I saw how to do centred labels in case you need to change something and it breaks
# https://stackoverflow.com/questions/71930367/matplotlib-center-align-y-tick-labels

# Note April 2026: changed the default font sizes for this notebook to the ones I use for publications, but then this plot got busted and I couldn't manually set some of the font sizes?
# so the fix is to change the default font sizes to my original screen-friendly smaller ones, make the plot, then go back to the bigger sizes at the end
SMALL_SIZE = 12
MEDIUM_SIZE = 12 
BIGGER_SIZE = 16 
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)    # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

#change this as needed to get FG names in desired format
refmt_res_labels = [f'{i[0]}*_{i[1]}' if i[0] in [57,58,59,60,63,73,74,75] else f'{i[0]}_{i[1]}' for i in FG_info.index.values]

fig = plt.figure(figsize=(10,9))

# left heatmap
# 3dp needed to be able to distinguish at level of single frames. but maybe that level of detail won't be necessary
ax1 = fig.add_subplot(121, axes_class=axisartist.Axes)
sns.heatmap(FG_info['% contacts'], ax=ax1, annot=True, fmt='.3f', cmap='Reds', cbar_kws={'location': 'top'},)
ax1.set_title('% frames with FG-ligand probe contacts')
ax1.axis["left",'top'].toggle(all=False)
ax1.yaxis.set_ticklabels([])
ax1.xaxis.set_ticklabels(['A/C*','B/A*','C/B*'])
ax1.axis[:].major_ticks.set_tick_out(True)

# right heatmap
ax2 = fig.add_subplot(122, axes_class=axisartist.Axes)
sns.heatmap(FG_info['% ind mods'], ax=ax2, annot=True, fmt='.3f', cmap='Blues', cbar_kws={'location': 'top'}, yticklabels=refmt_res_labels)
ax2.set_title('% QM-cluster models FG is in')
ax2.set_ylabel(None)
ax2.axis["left"].major_ticklabels.set_ha("center")
ax2.axis["right",'top'].toggle(all=False)
ax2.xaxis.set_ticklabels(['A/C*','B/A*','C/B*'])
ax2.axis[:].major_ticks.set_tick_out(True)

# this will need to be tweaked if you change y labels in any way!!!
plt.subplots_adjust(wspace=0.39)

# show plot before changing sizes back!
plt.show()

### reset sizes set up at the start of the notebook! otherwise subsequent plots will go back to small ###
SMALL_SIZE = 16
MEDIUM_SIZE = 18 
BIGGER_SIZE = 20 
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title


In [ ]:
### version of above with just % in each model and bigger text for paper ###

# nicer labels than above
refmt_res_labels = []
for i in FG_info.index.values:
    if i[0] in [57,58,59,60,63,73,74,75]:
        resi = str(i[0])+'*'
    else:
        resi = str(i[0])
    if i[1].startswith('SC'):
        refmt_res_labels.append(f'{resi} {i[1].split("_")[1].title()} {i[1].split("_")[0]}')
    else:
        refmt_res_labels.append(f'{resi} {i[1]}') 

fig, ax = plt.subplots(figsize=(scolw,12),layout='tight')

sns.heatmap(FG_info['% ind mods'], ax=ax, annot=True, fmt='.3f', cmap='Greens', cbar_kws={'location': 'top', 'label': '% of QM-cluster models FG is in'}, yticklabels=refmt_res_labels)
#ax.set_title('% of QM-cluster models FG is in')
ax.set_ylabel(None)
ax.xaxis.set_ticklabels(['A/C*','B/A*','C/B*'])
ax.xaxis.tick_top()

In [ ]:
refmt_res_labels

In [ ]:
### this is just here to help think about trends in charges!

charged_res = [(7,'SC_ARG'),(60,'SC_LYS'),(63,'SC_ARG'),(78,'SC_GLU'),(90,'SC_ARG'),(116,'SC_ARG')]
FG_info.loc[charged_res,'% ind mods']

## fig: FG stripe plots

I think this is gonna be way too busy to look at once all the data is filled in and not at all useful for the paper. But just leaving it in here as an idea.

In [ ]:
FG_df['fnum'] = [int(i[0].replace('f','')) for i in FG_df.index]

fig, ax = plt.subplots(figsize=(18,10),layout='constrained')

# plot data
for i, fg in enumerate(FG_info.index):
    ax.scatter(x=FG_df.loc[(slice(None),'A_ind'),'fnum'], y=FG_df.loc[(slice(None),'A_ind'),fg]*(i+0.8), s=1, c=tc.high_contrast.red)
    ax.scatter(x=FG_df.loc[(slice(None),'B_ind'),'fnum'], y=FG_df.loc[(slice(None),'B_ind'),fg]*(i+1), s=1, c=tc.high_contrast.yellow)
    ax.scatter(x=FG_df.loc[(slice(None),'C_ind'),'fnum'], y=FG_df.loc[(slice(None),'C_ind'),fg]*(i+1.2), s=1, c=tc.high_contrast.blue)
    #FG_df.loc[(slice(None),'A_ind'),:].plot(ax=axs[i],kind='scatter', x='fnum', y=fg, s=1, xlim=(0,20000))

# x axis options
ax.set_xlim(left=0,right=20000)

# y axis options
ax.set_ylim(top=0.5,bottom=len(FG_info.index)+0.5)
ax.set_yticks(range(1,len(FG_info.index)+1))
ax.set_yticklabels(FG_info.index)


## fig: histograms of size/charge/no FGs in each active site's set of models

In [ ]:
### histograms of values

activesites = ['A/C*','B/A*','C/B*']
cols = [tc.high_contrast.red,tc.high_contrast.yellow,tc.high_contrast.blue]
fig, axs = plt.subplots(nrows=3,ncols=4,figsize=(12,7),layout='constrained')
for i, mod in enumerate(['A_ind','B_ind','C_ind']):
    if mod in QMdata.keys():
        for j, yv in enumerate(['size','charge','nFG_prot','nFG_wat']):
            binrange = np.arange(np.floor(plotopts[yv]['min'])-0.5, np.ceil(plotopts[yv]['max'])+1.5,1)
            QMdata[mod].hist(column=yv,
                        ax=axs[i,j],
                        bins=binrange,
                        legend=False, 
                        grid=False,
                        color=cols[i])
            axs[i,j].set_title(None)
            if j == 0:
                axs[i,j].set_ylabel(activesites[i])
            else:
                axs[i,j].set_ylabel(None)
            if i == 2:
                axs[i,j].set_xlabel(plotopts[yv]['longlabel'])
            else:
                axs[i,j].set_xlabel(None)

# align ylims without sharey because that's fussy
for c in [0,1,2,3]:
    ylim_max=max([axs[r,c].get_ylim()[1] for r in [0,1,2]])
    for r in [0,1,2]:
        axs[r,c].set_ylim(bottom=0,top=ylim_max)

fig.align_labels()

# Results part 2: QM-cluster model energies

In [ ]:
# see how many data points have been collected
for i in ['A_ind','B_ind','C_ind']:
    print(f'{i}: {len(QMdata[i].loc[QMdata[i]["done"] == "Y",:])} out of {len(QMdata[i])}')

## fig: histograms of free energies

In [ ]:
### just energies for paper  ###

cols = [tc.high_contrast.red,tc.high_contrast.yellow,tc.high_contrast.blue]
fig, axs = plt.subplots(nrows=2,ncols=3,figsize=(dcolw,10),layout='constrained')
for j, mod in enumerate(['A_ind','B_ind','C_ind']):
    if mod in QMdata.keys():
        for i, yv in enumerate(['dGa','dGr']):
            binrange = np.arange(np.floor(plotopts[yv]['min']), np.ceil(plotopts[yv]['max'])+1,1)
            QMdata[mod].hist(column=yv,
                        ax=axs[i,j],
                        orientation='horizontal',
                        bins=binrange,
                        legend=False, 
                        grid=False,
                        color=cols[j])
            axs[i,j].set_title(None)
            if i == 1:
                axs[i,j].set_xlabel(active_sites[j])
            else:
                axs[i,j].set_xlabel(None)
                axs[i,j].set_title(active_sites[j])
            if j == 0:
                axs[i,j].set_ylabel(plotopts[yv]['longlabel'])
            else:
                axs[i,j].set_ylabel(None)
            axs[i,j].axhline(y=QMdata[mod][yv].mean(),color='black',linestyle='--', label=f'mean = {round(QMdata[mod][yv].mean(),1)} kcal/mol', zorder=0)
            axs[i,j].axhline(y=QMdata[mod][yv].median(),color='black',linestyle=':', label=f'median = {round(QMdata[mod][yv].median(),1)} kcal/mol', zorder=0)
            if yv == 'dGa': axs[i,j].legend(loc='upper right')
            elif yv == 'dGr': axs[1,j].legend(loc='lower right')

# align ylims without sharey because that's fussy
for r in [0,1]:
    xlim_max=max([axs[r,c].get_xlim()[1] for c in [0,1,2]])
    for c in [0,1,2]:
        axs[r,c].set_xlim(left=0,right=xlim_max)

fig.supxlabel('number of models')
fig.align_labels()

## fig: violin plots of free energies
(as alternative to histograms where you can see spread better)

In [ ]:
# seaborn violin plots look nicer than matplotlib and work even if your dataset has NaNs
# but also seaborn works best with long form tables so need to restructure the data first (explanation of long form vs wide form here: https://seaborn.pydata.org/tutorial/data_structure.html)
plotdf = pd.concat([QMdata['A_ind'][['fnum','ligand','dGa','dGr']],QMdata['B_ind'][['fnum','ligand','dGa','dGr']],QMdata['C_ind'][['fnum','ligand','dGa','dGr']]],ignore_index=True)

# fig setup
cols = [tc.high_contrast.red,tc.high_contrast.yellow,tc.high_contrast.blue]
fig, axs = plt.subplots(nrows=2,ncols=1,figsize=(8,8),layout='constrained')

for i, yv in enumerate(['dGa','dGr']):
    sns.violinplot(ax=axs[i], data=plotdf, x='ligand', y=yv, hue='ligand', palette=cols, density_norm='count', common_norm='true', zorder=2)
    axs[i].set_xlim(left=-0.5,right=2.5)
    axs[i].set_ylim(bottom=plotopts[yv]['min']-5, top=plotopts[yv]['max']+5)
    axs[i].set_xlabel(None)
    axs[i].set_ylabel(plotopts[yv]['longlabel'])
    axs[i].grid(axis='y', which='major', linestyle=':')
    axs[i].axhline(y=0, color='black', zorder=1)
    axs[i].set_axisbelow(True)
    # medians?
    axs[i].axhline(y=np.nanmedian(QMdata['A_ind'][yv]), color=cols[0], linestyle='--', zorder=1)
    axs[i].axhline(y=np.nanmedian(QMdata['B_ind'][yv]), color=cols[1], linestyle='--', zorder=1)
    axs[i].axhline(y=np.nanmedian(QMdata['C_ind'][yv]), color=cols[2], linestyle='--', zorder=1)

axs[0].set_xticklabels([])
axs[1].set_xticklabels(active_sites)
fig.supxlabel('active site')

fig.align_labels()

## table: summary stats

In [ ]:
# process dGa values first
df1 = pd.concat([QMdata['A_ind']['dGa'],QMdata['B_ind']['dGa'],QMdata['C_ind']['dGa']],axis=1)
df1.columns = ['A/C*','B/A*','C/B*']
df2 = df1.agg(func=['count','median','mean','std','max','min'])
# pandas std does ddof=1 by default so replace with ddof=0 version. for more detail see notes on this page: https://numpy.org/doc/2.1/reference/generated/numpy.std.html
df2.loc['std',:] = np.std(df1, axis=0, ddof=0)
df2.loc['range',:] = df2.loc['max',:]-df2.loc['min',:]
# add values for all active sites combined
df2['all'] = [sum(df1.count()), df1.mean(axis=None), df1.median(axis=None), np.nanstd(df1.values,ddof=0), df1.max(axis=None), df1.min(axis=None), df1.max(axis=None)-df1.min(axis=None)]
# copy to "df" to save for later
df = df2.copy(deep=True)

# repeat for dGr
df1 = pd.concat([QMdata['A_ind']['dGr'],QMdata['B_ind']['dGr'],QMdata['C_ind']['dGr']],axis=1)
df1.columns = ['A/C*','B/A*','C/B*']
df2 = df1.agg(func=['count','median','mean','std','max','min'])
df2.loc['std',:] = np.std(df1, axis=0, ddof=0)
df2.loc['range',:] = df2.loc['max',:]-df2.loc['min',:]
# add values for all active sites combined
df2['all'] = [sum(df1.count()), df1.mean(axis=None), df1.median(axis=None), np.nanstd(df1.values,ddof=0), df1.max(axis=None), df1.min(axis=None), df1.max(axis=None)-df1.min(axis=None)]

# combine dGa and dGr into one table
df = pd.concat([df,df2],axis=1,keys=[r'$\Delta G ^\ddagger$',r'$\Delta G _{rxn}$']).round(1)

# print latex table code
tabout = df.to_latex(caption='Summary statistics for computed free energies. All values in kcal/mol.',
                    label='tab:overallstats',
                    column_format='lcccccccc', 
                    multicolumn_format='c',
                    float_format="%.1f")
print(tabout)

## fig: free energies vs model features (size/charge/no. frags/etc)

In [ ]:
### plot each set of energies against size/charge/Nprot/Nwat

colsdark = [tc.medium_contrast.dark_red,tc.medium_contrast.dark_yellow,tc.medium_contrast.dark_blue]
colslight = [tc.medium_contrast.light_red,tc.medium_contrast.light_yellow,tc.medium_contrast.light_blue]

fig, axs = plt.subplots(nrows=3,ncols=4,figsize=(18,12),layout='constrained')

legend_elements = []

for i, mod in enumerate(['A_ind','B_ind','C_ind']):
    # define markers that will be used so it's easier to make one combined legend
    legend_elements.append(Line2D([0], [0], marker='o', ls='none', label=f"{plotopts['dGa']['shortlabel']} ({active_sites[i]})", color=colsdark[i], markersize=10))
    legend_elements.append(Line2D([0], [0], marker='^', ls='none', label=f"{plotopts['dGr']['shortlabel']} ({active_sites[i]})", color=colslight[i], markersize=10))
    if mod in QMdata.keys():
        for j, xv in enumerate(['size','charge','nFG_prot','nFG_wat']):
            QMdata[mod].plot(ax=axs[i,j], kind='scatter', x=xv, y='dGa', marker='o', c=colsdark[i], zorder=2)
            QMdata[mod].plot(ax=axs[i,j], kind='scatter', x=xv, y='dGr', marker='^', c=colslight[i], zorder=2)
            if xv == 'size':
                axs[i,j].set_xlim(left=plotopts[xv]['min']-5,right=plotopts[xv]['max']+5)
            elif xv == 'charge':
                axs[i,j].set_xlim(left=plotopts[xv]['min']-0.5,right=plotopts[xv]['max']+0.5)
            else:
                axs[i,j].set_xlim(left=plotopts[xv]['min']-1,right=plotopts[xv]['max']+1)
            axs[i,j].set_ylim(bottom=plotopts['dGr']['min']-3,top=plotopts['dGa']['max']+3)
            if j == 0:
                axs[i,j].set_ylabel(active_sites[i])
            else:
                axs[i,j].set_ylabel(None)
            if i == 2:
                axs[i,j].set_xlabel(plotopts[xv]['longlabel'])
            else:
                axs[i,j].set_xlabel(None)
            # add lines for medians
            axs[i,j].axhline(y=np.nanmedian(QMdata[mod]['dGa']), c='black', linestyle='--', zorder=1)
            axs[i,j].axhline(y=np.nanmedian(QMdata[mod]['dGr']), c='black', linestyle='--', zorder=1)
            # solid x-axis and dotted grid lines
            axs[i,j].axhline(y=0, c='black', zorder=1)
            axs[i,j].grid(axis='y', which='major', linestyle=':')
            axs[i,j].set_axisbelow(True)

# manually change x-ticks for Nwat plots because they're defaulting to 0.5s ugh
for i in [0,1,2]:
    axs[i,3].set_xticks([2,4,6,8,10,12])

# add in shared legend
fig.legend(handles=legend_elements,loc='outside upper center',ncols=6,handletextpad=0.1,columnspacing=2,borderaxespad=0.3)

fig.supylabel(r'$\Delta G$ (kcal/mol)')
fig.align_labels()



## idea: relate energies to specific active site fragments. use presence in model/contact info originally collected

replaced original (bad) idea of violins with table of means with and without FG - maybe turn into heatmap? would need to use numerical values instead of "mean +/- std" strings. or can just be table in SI or something

In [ ]:
new_FG_df = FG_df.swaplevel()
new_FG_df = new_FG_df.drop(columns=[('fnum', '')])

newdict = {}
for lig in ['A_ind','B_ind','C_ind']:
    newdict[('# models',lig)] = {}
    newdict[('# models',lig)] = {}
    for i in ['dGa','dGr']:
        newdict[(i,lig)] = {}
        newdict[(i,lig)] = {}

    newdict[('# models',lig)][('all','')] = int(len(QMdata[lig]))
    newdict[('dGa',lig)][('all','')] = f"{QMdata[lig]['dGa'].mean().round(1)} ± {QMdata[lig]['dGa'].std(ddof=0).round(1)}"
    newdict[('dGr',lig)][('all','')] = f"{QMdata[lig]['dGr'].mean().round(1)} ± {QMdata[lig]['dGr'].std(ddof=0).round(1)}"

    for fg in new_FG_df.columns:
        fgstr = f'{fg[0]}_{fg[1]}'
        selrows_with = new_FG_df.loc[lig,fg] == 1
        if selrows_with.sum() > 0:
            newdict[('# models',lig)][(fgstr,'with')] = int(selrows_with.sum())
            newdict[('dGa',lig)][(fgstr,'with')] = f"{QMdata[lig].loc[selrows_with,'dGa'].mean().round(1)} ± {QMdata[lig].loc[selrows_with,'dGa'].std(ddof=0).round(1)}"
            newdict[('dGr',lig)][(fgstr,'with')] = f"{QMdata[lig].loc[selrows_with,'dGr'].mean().round(1)} ± {QMdata[lig].loc[selrows_with,'dGr'].std(ddof=0).round(1)}"
        else:
            newdict[('# models',lig)][(fgstr,'with')] = 0
            newdict[('dGa',lig)][(fgstr,'with')] = ''
            newdict[('dGr',lig)][(fgstr,'with')] = ''
        selrows_without = new_FG_df.loc[lig,fg] != 1
        if selrows_without.sum() > 0:
            newdict[('# models',lig)][(fgstr,'without')] = int(selrows_without.sum())
            newdict[('dGa',lig)][(fgstr,'without')] = f"{QMdata[lig].loc[selrows_without,'dGa'].mean().round(1)} ± {QMdata[lig].loc[selrows_without,'dGa'].std(ddof=0).round(1)}"
            newdict[('dGr',lig)][(fgstr,'without')] = f"{QMdata[lig].loc[selrows_without,'dGr'].mean().round(1)} ± {QMdata[lig].loc[selrows_without,'dGr'].std(ddof=0).round(1)}"
        else:
            newdict[('# models',lig)][(fgstr,'without')] = 0
            newdict[('dGa',lig)][(fgstr,'without')] = ''
            newdict[('dGr',lig)][(fgstr,'without')] = ''


means_with_without_FGs = pd.DataFrame.from_dict(newdict)
means_with_without_FGs = means_with_without_FGs.sort_index(axis=1).round(2)
means_with_without_FGs

## fig: processing/plots for Nate's poster

keeping this here in case people want to keep using it on subsequent posters - will update with latest data each time notebook is run

also serves as a good example of brute forcing a plot to look how you want. also demonstrates a situation where I'd use matplotlib violins over seaborn - I could not get lined up/overlaid half violins to work at all with seaborn, but it's so easy to pick the x positions and which half of the violin to plot with matplotlib violinplot

In [ ]:
[c for c in QMdata['A_ind'].columns if 'maxmove_diff' in c]

In [ ]:
plotdf1 = pd.concat([QMdata['A_ind'], QMdata['B_ind'], QMdata['C_ind']])
# filter out anything that's potentially bad: only want ones that don't have any lig/water rmsds >1, and don't have any too small imag modes
selrows = plotdf1['done'] == 'Y'
for c in ['rms_ts-r_lig', 'rms_ts-r_wat', 'rms_p-r_lig', 'rms_p-r_wat', 'rms_ts-p_lig', 'rms_ts-p_wat']:
    selrows = selrows & (plotdf1[c] < 1)
for c in ['maxmove_diff_tmp-init','maxmove_diff_guess-ts','maxmove_diff_ts-r','maxmove_diff_ts-p','maxmove_diff_p-r','maxmove_diff_r-init']:
    selrows = selrows & (plotdf1[c] < 2)
selrows = selrows & (plotdf1['ts_maximagmode']<-350)
plotdf1 = plotdf1.loc[selrows,:]

def std(x):
    return x.std(ddof=0)

# values for Nate to put on poster
plotdf1['dGa'].apply(['mean',std])

In [ ]:
import matplotlib.patches as mpatches

##### plot #####
cols = [tc.high_contrast.red,tc.high_contrast.yellow,tc.high_contrast.blue]
fig, ax = plt.subplots(figsize=(8,8),layout='constrained')

dfs = []
for i, key in enumerate(['A_ind','B_ind','C_ind']):
    selrows = QMdata[key]['done'] == 'Y'
    for c in ['rms_ts-r_lig', 'rms_ts-r_wat', 'rms_p-r_lig', 'rms_p-r_wat', 'rms_ts-p_lig', 'rms_ts-p_wat']:
        selrows = selrows & (QMdata[key][c] < 1)
    selrows = selrows & (QMdata[key]['ts_maximagmode']<-350)
    plotdf = QMdata[key].loc[selrows,:]
    dfs.append(plotdf)
    # first "layer" = semi-transparent fill
    parts1 = ax.violinplot(plotdf['dGa'],positions=[1],side='low', showextrema=False, showmeans=False) #widths=0.2, 
    for pc in parts1['bodies']:
        pc.set_alpha(0.3)
        pc.set_edgecolor(cols[i])
        pc.set_facecolor(cols[i])
        pc.set_zorder(4-i) #layer in reverse order - blue is last but darkest so harder to see everything when it's on top
    # second layer: opaque outlines, no fill
    parts2 = ax.violinplot(plotdf['dGa'],positions=[1],side='low', showextrema=False, showmeans=True) #widths=0.2, 
    for pc in parts2['bodies']:
        pc.set_alpha(1)
        pc.set_linewidth(2)
        pc.set_edgecolor(cols[i])
        pc.set_facecolor("None")
        pc.set_zorder(4-i)
    for partname in ['cmeans']:
        vp = parts2[partname]
        vp.set_edgecolor(cols[i])
        vp.set_facecolor(cols[i])
        vp.set_linewidth(2)
        vp.set_zorder(4-i)

#allcol = 'black'
allcol = '#008816'
plotdf = pd.concat(dfs)
parts1 = ax.violinplot(plotdf['dGa'],positions=[1],side='high', showextrema=False, showmeans=False) #widths=0.6, 
for pc in parts1['bodies']:
    pc.set_alpha(0.3)
    pc.set_edgecolor(allcol)
    pc.set_facecolor(allcol)
    pc.set_zorder(5)    # topmost layer
parts2 = ax.violinplot(plotdf['dGa'],positions=[1],side='high', showextrema=False, showmeans=True) #widths=0.6, 
for pc in parts2['bodies']:
    pc.set_alpha(1)
    pc.set_edgecolor(allcol)
    pc.set_facecolor("none")
    pc.set_linewidth(2)
    pc.set_zorder(5)
for partname in ['cmeans']:
    vp = parts2[partname]
    vp.set_edgecolor(allcol)
    vp.set_facecolor(allcol)
    vp.set_linewidth(2)
    vp.set_zorder(5)

# create example opaque outline/translucent fill rectangles and use as legend icons
patches=[0,0,0,0]
for i, c in enumerate(cols):
    patches[i] = mpatches.Patch(facecolor=(c,0.2), edgecolor=(c,1), linewidth=2, label=activesites[i])
patches[3] = mpatches.Patch(facecolor=(allcol,0.2), edgecolor=(allcol,1), linewidth=2, label='all')
plt.legend(handles=patches, loc='upper right', title='active site', fontsize=20, title_fontsize=20)

ax.axhline(y=15.4, color='black', linestyle='--', zorder=1, linewidth=2)
ax.annotate(r'expt. $\Delta G ^\ddagger$',xy=(1.48,15.6),ha='right',va='bottom', fontsize=20)

ax.set_xlim(left=0.7,right=1.5)
ax.set_ylim(bottom=10,top=30)
ax.set_xlabel(None)
ax.set_ylabel(plotopts['dGa']['longlabel'], fontsize=20)
ax.grid(axis='y', which='major', linestyle=':')
ax.set_axisbelow(True)
ax.set_xticks([])
ax.set_xticklabels([])

fig.align_labels()

# Results part 3: Sampling strategies

<span style="color:red">**Currently for demonstration only!! TALK TO DOMI BEFORE EDITING/USING!!!!!**</span>

**EVERYTHING IN THIS BLOCK CURRENTLY USES FAKE (RANDOMLY GENERATED) DATA AND SHOULD NOT BE USED BEFORE ALL THE RESULTS ARE COLLECTED.** When you're ready to make these plots for real, all the code blocks for the plots and tables in this section will need to be changed to use the values from QMdata. **At this point, remove the code block below that generates the random data.** It is very easy to miss a spot when changing a lot of lines of code, and we don't want the old fake values accidentally getting used in any of the real plots. If the randdata dict doesn't exist, you'll get errors any time it's called which'll make it easy to find anything you've missed.

## prep: generate random data

Obviously for the paper you will use the actual data from the QMdata dictionary. But I'm just setting things up with some randomly generated data to demonstrate the analysis I want you to do. 

In [27]:
randdata = {'A:128': pd.DataFrame([list(range(1,20001,1)), ['A:128' for i in range(1,20001)], np.random.normal(15, 3, 20000),np.random.normal(-15, 3, 20000)],index=['fnum','ligand','dGa','dGr'],columns=[f'f{str(i).zfill(5)}' for i in range(1,20001,1)]).T,
            'B:256': pd.DataFrame([list(range(1,20001,1)), ['B:256' for i in range(1,20001)], np.random.normal(16, 3, 20000),np.random.normal(-12, 3, 20000)],index=['fnum','ligand','dGa','dGr'],columns=[f'f{str(i).zfill(5)}' for i in range(1,20001,1)]).T,
            'C:384': pd.DataFrame([list(range(1,20001,1)), ['C:384' for i in range(1,20001)], np.random.normal(15.5, 2, 20000),np.random.normal(-14, 2, 20000)],index=['fnum','ligand','dGa','dGr'],columns=[f'f{str(i).zfill(5)}' for i in range(1,20001,1)]).T}

## R3a) Atsu's original subsets

REMEMBER: Atsu's paper only looked at A/C* (A:128) active site, so this analysis should only be done on Dr D's QM-A data because most of his sampling schemes were active site specific!!!

### fig: violin plots of free energies

In [ ]:
# select data
dfs = []
for Si in [key for key in old_subsets.keys() if key != 'all_250']:
    nextdf = randdata['A:128'].loc[old_subsets[Si],:].copy(deep=True)
    nextdf['original subset'] = [Si for j in range(0,len(nextdf))]
    dfs.append(nextdf)
plotdf = pd.concat(dfs)
# reformat in same way for all 250 frames from old subsets
plotdf_2 = randdata['A:128'].loc[old_subsets['all_250'],:].copy(deep=True)
plotdf_2['original subset'] = ['all 250' for j in range(0,len(plotdf_2))]
# and again for all data from all 20000 frames
plotdf_all = randdata['A:128'].loc[:,:].copy(deep=True)
plotdf_all['original subset'] = ['full set' for j in range(0,len(plotdf_all))]

fig = plt.figure(figsize=(18,8))
gs = matplotlib.gridspec.GridSpec(2, 3, height_ratios=[1,1], width_ratios=[9,1,1], left=0.05, right=0.48, wspace=0.05, hspace=0.05)
axs = [fig.add_subplot(gs[i]) for i in [0,1,2,3,4,5]]

# all original subset violins
sns.violinplot(ax=axs[0], data=plotdf, x='original subset', y='dGa', density_norm='count', common_norm=True)
sns.violinplot(ax=axs[3], data=plotdf, x='original subset', y='dGr', density_norm='count', common_norm=True)

# full 250 frame set
sns.violinplot(ax=axs[1], data=plotdf_2, x='original subset', y='dGa')
sns.violinplot(ax=axs[4], data=plotdf_2, x='original subset', y='dGr')

# all 20000 models
sns.violinplot(ax=axs[2], data=plotdf_all, x='original subset', y='dGa')
sns.violinplot(ax=axs[5], data=plotdf_all, x='original subset', y='dGr')

for i in [0,1,2,3,4,5]:
    axs[i].set_xlabel(None)
    axs[i].grid(axis='y',which='major',linestyle=':',color='gray')
    if i in [0,1,2]:
        axs[i].set_ylim(bottom=randdata['A:128']['dGa'].min()-2,top=randdata['A:128']['dGa'].max()+2)
        axs[i].set_xticklabels([])
    else:
        axs[i].set_ylim(bottom=randdata['A:128']['dGr'].min()-2,top=randdata['A:128']['dGr'].max()+2)
    if i not in [0,3]:
        axs[i].set_ylabel(None)
        axs[i].set_yticklabels([])

axs[0].set_ylabel(plotopts['dGa']['longlabel'])
axs[3].set_ylabel(plotopts['dGr']['longlabel'])

fig.align_labels()

### table: subset stats

In [ ]:
old_subset_stats = {}
for Si in old_subsets.keys():
    old_subset_stats[Si] = {}
    for g in ['dGa','dGr']:
        old_subset_stats[Si][f'{g}_mean'] = randdata['A:128'].loc[old_subsets[Si],g].mean()
        old_subset_stats[Si][f'{g}_std'] = randdata['A:128'].loc[old_subsets[Si],g].std(ddof=0)
old_subset_stats['all_20000'] = {}
for g in ['dGa','dGr']:
    old_subset_stats['all_20000'][f'{g}_mean'] = randdata['A:128'].loc[:,g].mean()
    old_subset_stats['all_20000'][f'{g}_std'] = randdata['A:128'].loc[:,g].std(ddof=0)

df = pd.DataFrame.from_dict(old_subset_stats,orient='index')
### print at this point just to look at in notebook ###
#print(round(df,2))

df.columns = [r'mean $\Delta G ^\ddagger$', r'$ \sigma \Delta G ^\ddagger$', r'mean $\Delta G _{rxn}$', r'$ \sigma \Delta G _{rxn}$']
tabout = df.to_latex(caption='Summary statistics for subsets used in previous paper. All values in kcal/mol.',
                    label='tab:old-subset-stats',
                    column_format='lllll', 
                    multicolumn_format='c',
                    float_format="%.1f")
### or print here to get proper latex code ###
print(tabout)

## R3b) examples of what to do with new schemes/subsets

### comparing all replicas of one sampling scheme/subset size combination

In [ ]:
# select data
dfs = []
for i in range(0,len(sampled_sets['heavy_toxtal_cluster']['sample_30'])):
    nextdf = randdata['A:128'].loc[sampled_sets['heavy_toxtal_cluster']['sample_30'][i],:].copy(deep=True)
    nextdf['replica'] = [f'sample_30_{i+1}' for j in range(0,len(nextdf))]
    dfs.append(nextdf)
plotdf = pd.concat(dfs)

# data for all models
plotdf_all = randdata['A:128'].loc[:,:].copy(deep=True)
plotdf_all['replica'] = ['full set' for j in range(0,len(plotdf_all))]

# set up plot
fig = plt.figure(figsize=(18,8))
gs = matplotlib.gridspec.GridSpec(2, 2, height_ratios=[1,1], width_ratios=[len(dfs),1], left=0.05, right=0.48, wspace=0.05, hspace=0.05)
axs = [fig.add_subplot(gs[i]) for i in [0,1,2,3]]

# subset violins
sns.violinplot(ax=axs[0], data=plotdf, x='replica', y='dGa', density_norm='count', common_norm=True)
sns.violinplot(ax=axs[2], data=plotdf, x='replica', y='dGr', density_norm='count', common_norm=True)

# full set violins
sns.violinplot(ax=axs[1], data=plotdf_all, x='replica', y='dGa')
sns.violinplot(ax=axs[3], data=plotdf_all, x='replica', y='dGr')

for i in [0,1,2,3]:
    axs[i].set_xlabel(None)
    axs[i].grid(axis='y',which='major',linestyle=':',color='gray')
    if i in [0,1]:
        axs[i].set_ylim(bottom=randdata['A:128']['dGa'].min()-2,top=randdata['A:128']['dGa'].max()+2)
        axs[i].set_xticklabels([])
    else:
        axs[i].set_ylim(bottom=randdata['A:128']['dGr'].min()-2,top=randdata['A:128']['dGr'].max()+2)
    if i not in [0,2]:
        axs[i].set_ylabel(None)
        axs[i].set_yticklabels([])

axs[0].set_ylabel(plotopts['dGa']['longlabel'])
axs[2].set_ylabel(plotopts['dGr']['longlabel'])
axs[2].set_xlabel('replica')
axs[2].tick_params(axis='x', labelrotation=90)
axs[3].tick_params(axis='x', labelrotation=90)

fig.align_labels()

<a id='subset_size_violins'></a>

### compare subset sizes for one sampling scheme using first replica of each size


In [ ]:
# select data
dfs = []
for key in sampled_sets['allMC_toxtal'].keys():
    nextdf = randdata['A:128'].loc[sampled_sets['allMC_toxtal'][key][0],:].copy(deep=True)
    nextdf['samplesize'] = [key.replace('sample_','') for j in range(0,len(nextdf))]
    dfs.append(nextdf)
plotdf = pd.concat(dfs)

# data for all models
plotdf_all = randdata['A:128'].loc[:,:].copy(deep=True)
plotdf_all['samplesize'] = ['full set' for j in range(0,len(plotdf_all))]

# set up plot
fig = plt.figure(figsize=(18,8))
gs = matplotlib.gridspec.GridSpec(2, 2, height_ratios=[1,1], width_ratios=[len(dfs),1], left=0.05, right=0.48, wspace=0.05, hspace=0.05)
axs = [fig.add_subplot(gs[i]) for i in [0,1,2,3]]

# subset violins
sns.violinplot(ax=axs[0], data=plotdf, x='samplesize', y='dGa', density_norm='count', common_norm=True)
sns.violinplot(ax=axs[2], data=plotdf, x='samplesize', y='dGr', density_norm='count', common_norm=True)

# full set violins
sns.violinplot(ax=axs[1], data=plotdf_all, x='samplesize', y='dGa')
sns.violinplot(ax=axs[3], data=plotdf_all, x='samplesize', y='dGr')

for i in [0,1,2,3]:
    axs[i].set_xlabel(None)
    axs[i].grid(axis='y',which='major',linestyle=':',color='gray')
    if i in [0,1]:
        axs[i].set_ylim(bottom=randdata['A:128']['dGa'].min()-2,top=randdata['A:128']['dGa'].max()+2)
        axs[i].set_xticklabels([])
    else:
        axs[i].set_ylim(bottom=randdata['A:128']['dGr'].min()-2,top=randdata['A:128']['dGr'].max()+2)
    if i not in [0,2]:
        axs[i].set_ylabel(None)
        axs[i].set_yticklabels([])

axs[0].set_ylabel(plotopts['dGa']['longlabel'])
axs[2].set_ylabel(plotopts['dGr']['longlabel'])
axs[2].set_xlabel('subset size')
axs[2].tick_params(axis='x', labelrotation=90)
axs[3].tick_params(axis='x', labelrotation=90)

fig.align_labels()

### prob don't want to plot this full range of sizes at once because the smallest subsets are too narrow... if we are doing this we should get rid of inner boxplot maybe?

### compare schemes with same subset size/using first replica of each


In [ ]:
# select data
dfs = []
for key in [i for i in sampled_sets.keys() if i != 'interval_sampling' and not i.startswith('activeB') and not i.startswith('activeC')]:
    nextdf = randdata['A:128'].loc[sampled_sets[key]['sample_150'][0],:].copy(deep=True)
    nextdf['scheme'] = [key for j in range(0,len(nextdf))]
    dfs.append(nextdf)
plotdf = pd.concat(dfs)

# data for all models
plotdf_all = randdata['A:128'].loc[:,:].copy(deep=True)
plotdf_all['scheme'] = ['full set' for j in range(0,len(plotdf_all))]

# set up plot
fig = plt.figure(figsize=(18,8))
gs = matplotlib.gridspec.GridSpec(2, 2, height_ratios=[1,1], width_ratios=[len(dfs),1], left=0.05, right=0.48, wspace=0.05, hspace=0.05)
axs = [fig.add_subplot(gs[i]) for i in [0,1,2,3]]

# subset violins
sns.violinplot(ax=axs[0], data=plotdf, x='scheme', y='dGa', density_norm='count', common_norm=True)
sns.violinplot(ax=axs[2], data=plotdf, x='scheme', y='dGr', density_norm='count', common_norm=True)

# full set violins
sns.violinplot(ax=axs[1], data=plotdf_all, x='scheme', y='dGa')
sns.violinplot(ax=axs[3], data=plotdf_all, x='scheme', y='dGr')

for i in [0,1,2,3]:
    axs[i].set_xlabel(None)
    axs[i].grid(axis='y',which='major',linestyle=':',color='gray')
    if i in [0,1]:
        axs[i].set_ylim(bottom=randdata['A:128']['dGa'].min()-2,top=randdata['A:128']['dGa'].max()+2)
        axs[i].set_xticklabels([])
    else:
        axs[i].set_ylim(bottom=randdata['A:128']['dGr'].min()-2,top=randdata['A:128']['dGr'].max()+2)
    if i not in [0,2]:
        axs[i].set_ylabel(None)
        axs[i].set_yticklabels([])

axs[0].set_ylabel(plotopts['dGa']['longlabel'])
axs[2].set_ylabel(plotopts['dGr']['longlabel'])
axs[2].set_xlabel('sampling scheme')
axs[2].tick_params(axis='x', labelrotation=90)
axs[3].tick_params(axis='x', labelrotation=90)

fig.align_labels()


list of everything to calculate:

- mean/median/std/max/min/range for each scheme/replica/size/active site
- spread of values for each (grouped across replicas)


### calculate subset stats

In [33]:
# some schemes can be used for all active sites because they're based on the entire trimer
generalschemes = ['interval_sampling', 'allMC_tofirst', 'allMC_toxtal', 'activeMC_tofirst', 'activeMC_toxtal', 'activeSC_tofirst', 'activeSC_toxtal', 'heavy_tofirst', 'heavy_toxtal', 'heavy_tofirst_cluster', 'heavy_toxtal_cluster']
# others are based on a specific active site's residues, so should only be used to sample from that active site's models
activeAschemes = ['activeA_MC_tofirst', 'activeA_MC_toxtal', 'activeA_SC_tofirst', 'activeA_SC_toxtal']
activeBschemes = ['activeB_MC_tofirst', 'activeB_MC_toxtal', 'activeB_SC_tofirst', 'activeB_SC_toxtal']
activeCschemes = ['activeC_MC_tofirst', 'activeC_MC_toxtal', 'activeC_SC_tofirst', 'activeC_SC_toxtal']

# going to put all the stats in a dictionary that can then be turned into a dataframe
subset_stats = {}

#  get stats for the full set first so the schemes can be compared to it
for lig in ['A:128','B:256','C:384']:
    subset_stats[('full set','',lig)] = {'dGa_mean': [], 'dGa_std': [], 'dGa_range': [], 'dGr_mean': [], 'dGr_std': [], 'dGr_range': []}
    for g in ['dGa','dGr']:
        subset_stats[('full set','',lig)][f'{g}_mean'].append(randdata[lig].loc[:,g].mean())
        subset_stats[('full set','',lig)][f'{g}_std'].append(randdata[lig].loc[:,g].std(ddof=0))
        subset_stats[('full set','',lig)][f'{g}_range'].append(np.ptp(randdata[lig].loc[:,g]))

# stats for general schemes
for scheme in generalschemes:
    for subset in sampled_sets[scheme].keys():
        subset_stats[(scheme,subset,'A:128')] = {'dGa_mean': [], 'dGa_std': [], 'dGa_range': [], 'dGr_mean': [], 'dGr_std': [], 'dGr_range': []}
        subset_stats[(scheme,subset,'B:256')] = {'dGa_mean': [], 'dGa_std': [], 'dGa_range': [], 'dGr_mean': [], 'dGr_std': [], 'dGr_range': []}
        subset_stats[(scheme,subset,'C:384')] = {'dGa_mean': [], 'dGa_std': [], 'dGa_range': [], 'dGr_mean': [], 'dGr_std': [], 'dGr_range': []}
        for replica in sampled_sets[scheme][subset]:
            for lig in ['A:128','B:256','C:384']:
                for g in ['dGa','dGr']:
                    subset_stats[(scheme,subset,lig)][f'{g}_mean'].append(randdata[lig].loc[replica,g].mean())
                    subset_stats[(scheme,subset,lig)][f'{g}_std'].append(randdata[lig].loc[replica,g].std(ddof=0))
                    subset_stats[(scheme,subset,lig)][f'{g}_range'].append(np.ptp(randdata[lig].loc[replica,g]))

# stats for each active site's specific schemes
for scheme in activeAschemes:
    for subset in sampled_sets[scheme].keys():
        subset_stats[(scheme,subset,'A:128')] = {'dGa_mean': [], 'dGa_std': [], 'dGa_range': [], 'dGr_mean': [], 'dGr_std': [], 'dGr_range': []}
        for replica in sampled_sets[scheme][subset]:
            for g in ['dGa','dGr']:
                subset_stats[(scheme,subset,'A:128')][f'{g}_mean'].append(randdata['A:128'].loc[replica,g].mean())
                subset_stats[(scheme,subset,'A:128')][f'{g}_std'].append(randdata['A:128'].loc[replica,g].std(ddof=0))
                subset_stats[(scheme,subset,'A:128')][f'{g}_range'].append(np.ptp(randdata['A:128'].loc[replica,g]))
for scheme in activeBschemes:
    for subset in sampled_sets[scheme].keys():
        subset_stats[(scheme,subset,'B:256')] = {'dGa_mean': [], 'dGa_std': [], 'dGa_range': [], 'dGr_mean': [], 'dGr_std': [], 'dGr_range': []}
        for replica in sampled_sets[scheme][subset]:
            for g in ['dGa','dGr']:
                subset_stats[(scheme,subset,'B:256')][f'{g}_mean'].append(randdata['B:256'].loc[replica,g].mean())
                subset_stats[(scheme,subset,'B:256')][f'{g}_std'].append(randdata['B:256'].loc[replica,g].std(ddof=0))
                subset_stats[(scheme,subset,'B:256')][f'{g}_range'].append(np.ptp(randdata['B:256'].loc[replica,g]))
for scheme in activeCschemes:
    for subset in sampled_sets[scheme].keys():
        subset_stats[(scheme,subset,'C:384')] = {'dGa_mean': [], 'dGa_std': [], 'dGa_range': [], 'dGr_mean': [], 'dGr_std': [], 'dGr_range': []}
        for replica in sampled_sets[scheme][subset]:
            for g in ['dGa','dGr']:
                subset_stats[(scheme,subset,'C:384')][f'{g}_mean'].append(randdata['C:384'].loc[replica,g].mean())
                subset_stats[(scheme,subset,'C:384')][f'{g}_std'].append(randdata['C:384'].loc[replica,g].std(ddof=0))
                subset_stats[(scheme,subset,'C:384')][f'{g}_range'].append(np.ptp(randdata['C:384'].loc[replica,g]))


In [ ]:
pd.DataFrame.from_dict(subset_stats,orient='index')

In [35]:
### turn lists of each replica's stats into "(min, max) = range" or medians
stats_ranges = {}
stats_medians = {}

# add in full set stats as is
for lig in ['A:128','B:256','C:384']:
    stats_ranges[('full set','',lig)] = {}
    stats_medians[('full set','',lig)] = {}
    for key2 in subset_stats[('full set','',lig)].keys():
        stats_ranges[('full set','',lig)][key2] = f'{subset_stats[("full set","",lig)][key2][0]:.1f}'
        stats_medians[('full set','',lig)][key2] = f'{subset_stats[("full set","",lig)][key2][0]:.2f}'

# for all other (scheme,subset,ligand) tuples
for key1 in [i for i in subset_stats.keys() if i[0] != 'full set']:
    stats_ranges[key1] = {}
    stats_medians[key1] = {}
    # for each combination of dGa/dGr and mean/std/range
    for key2 in subset_stats[key1].keys():
        # get (min,max) range of stat values rounded to 2 d.p.
        stats_ranges[key1][key2] = f'({min(subset_stats[key1][key2]):.1f}, {max(subset_stats[key1][key2]):.1f})'
        #stats_ranges[key1][key2] = f'({min(subset_stats[key1][key2]):.1f}, {max(subset_stats[key1][key2]):.1f}) = {np.ptp(subset_stats[key1][key2]):.1f}'
        stats_medians[key1][key2] = f'{np.median(subset_stats[key1][key2]):.2f}'

# turn dicts into dataframes
stat_range_df = pd.DataFrame.from_dict(stats_ranges,orient='index')
stat_median_df = pd.DataFrame.from_dict(stats_medians,orient='index')
# this replaces subset text label with integer size of sample for better ordering!
idx_with_sample_sizes = [(i[0],20000,i[2]) if i[0] == 'full set' 
 else (i[0],20,i[2]) if i[1] == 'every1000th' 
 else (i[0],40,i[2]) if i[1] == 'every500th' 
 else (i[0],100,i[2]) if i[1] == 'every200th' 
 else (i[0],200,i[2]) if i[1] == 'every100th' 
 else (i[0],int(i[1].replace('sample_','')),i[2])
 for i in list(stat_range_df.index.values)]
stat_range_df.index = pd.MultiIndex.from_tuples(idx_with_sample_sizes,names=['scheme','subset_size','ligand'])
stat_median_df.index = pd.MultiIndex.from_tuples(idx_with_sample_sizes,names=['scheme','subset_size','ligand'])
# now sort by ligand first so schemes for same active site can be compared more easily
stat_range_df.sort_index(axis=0, level=['ligand','scheme','subset_size'], inplace=True)
stat_median_df.sort_index(axis=0, level=['ligand','scheme','subset_size'], inplace=True)
# make columns multiindex as well
stat_range_df.columns = pd.MultiIndex.from_tuples([(i.split('_')[0],i.split('_')[1]) for i in stat_range_df.columns])
stat_median_df.columns = pd.MultiIndex.from_tuples([(i.split('_')[0],i.split('_')[1]) for i in stat_median_df.columns])

# make all activeA/activeB/activeC subsets have same "ind_active" label
newidxlist = []
for i in stat_range_df.index.values:
    newidx = list(i)
    if newidx[0].startswith('activeA'):
        newidx[0] = newidx[0].replace('activeA','own_active')
    elif newidx[0].startswith('activeB'):
        newidx[0] = newidx[0].replace('activeB','own_active')
    elif newidx[0].startswith('activeC'):
        newidx[0] = newidx[0].replace('activeC','own_active')
    elif newidx[0].startswith('activeMC'):
        newidx[0] = newidx[0].replace('activeMC','all_active_MC')
    elif newidx[0].startswith('activeSC'):
        newidx[0] = newidx[0].replace('activeSC','all_active_SC')
    newidxlist.append(tuple(newidx))
stat_range_df.index = pd.MultiIndex.from_tuples(newidxlist)
stat_median_df.index = pd.MultiIndex.from_tuples(newidxlist)

# now reshape so active sites are on columns
stat_range_df = stat_range_df.unstack(level=2)
stat_range_df = stat_range_df.loc[:,stat_range_df.columns.drop('range',level=1)]
stat_range_df = stat_range_df.reorder_levels(order=[2,0,1],axis='columns')
stat_range_df.sort_index(axis='columns',level=[0,1,2],inplace=True)

stat_median_df = stat_median_df.unstack(level=2)
stat_median_df = stat_median_df.loc[:,stat_median_df.columns.drop('range',level=1)]
stat_median_df = stat_median_df.reorder_levels(order=[2,0,1],axis='columns')
stat_median_df.sort_index(axis='columns',level=[0,1,2],inplace=True)


In [ ]:
# get full list of schemes as listed in stat_median_df with: stat_median_df.index.get_level_values(0).unique()
schemes_to_show = ['full set', 'interval_sampling', 'allMC_toxtal', 'all_active_MC_toxtal', 'own_active_MC_toxtal', 'all_active_SC_toxtal', 'own_active_SC_toxtal', 'heavy_toxtal', 'heavy_toxtal_cluster']

#with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#    print(stat_median_df.loc[schemes_to_show,:])

tabout = stat_median_df.loc[schemes_to_show,:].to_latex(caption='PLACEHOLDER WITH FAKE DATA: Summary statistics for sampled subsets, given as median from a set of 10 replicas. All schemes except interval sampling are based on RMSDs with respect to the X-ray crystal structure. All values in kcal/mol.',
                    label='tab:new-subset-stats',
                    column_format='llcccccccccccc', 
                    multicolumn_format='c')
print(tabout)